# T4 (CUDA) → сравнение с локальным llama.cpp

Ручной прогон для **Google Colab на GPU T4** — зеркало нагрузочного теста из ДЗ
`09_l26_triton_deploy` (`perf/load_test.py` + `llama-bench`).

Сравниваем **одну и ту же модель** `Qwen2.5-0.5B-Instruct Q4_K_M` (0.5B) на трёх
llama.cpp-бэкендах:

| Бэкенд | Где | ggml |
|--------|-----|------|
| **CUDA (T4)** | этот ноутбук (Colab NVIDIA T4) | ggml-cuda |
| Vulkan (Radeon 780M, AMD 7840HS) | локально | ggml-vulkan |
| CPU (ggml-cpu, без Vulkan) | локально | ggml-cpu |

Методика — та же, что в разделе 4 `REPORT.md`:
- `llama-bench`: `-p 512 -n 128 -t 4 -r 3` → ток/с (prefill pp512 / generation tg128);
- HTTP load-test: `concurrency=2, duration=30, warmup=2`, `/completion`
  с `n_predict=128, temperature=0.1` → rps / p50 / p95 / p99.

Локальные цифры подставлены из последнего прогона (CPU 4 потока / Vulkan 780M);
в конце их всегда можно перезаписать своими.

> Запуск: `Runtime → Change runtime type → T4 GPU`, затем `Runtime → Run all`.
> Сборка llama.cpp с CUDA занимает ~5–10 минут, дальше всё считает за ~2 минуты.


In [ ]:
import subprocess, shutil

# --- 0. Проверка GPU -----------------------------------------------
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv"], capture_output=True, text=True).stdout)
for tool in ("cmake", "ninja", "git", "g++"):
    print(f"{tool:6s}", shutil.which(tool))


In [ ]:
import subprocess, shutil

# --- 1. Тулчейн (в Colab обычно уже есть, докидываем при отсутствии) ----
need = [t for t in ("cmake", "ninja", "git") if shutil.which(t) is None]
if need:
    subprocess.run(["apt-get", "install", "-y", "-qq", "cmake", "ninja-build", "git"],
                   check=True)
print({t: shutil.which(t) for t in ("cmake", "ninja", "git")})


### 2. Сборка llama.cpp с CUDA

Пинаем **тот же коммит `0b1bad14f` (llama.cpp 0.19.0)**, который использовался в
локальных замерах — чтобы comparisons были apples-to-apples. T4 = sm_75.


In [ ]:
import os, subprocess

ROOT = os.path.abspath("llama.cpp")
COMMIT = "0b1bad14f"

if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--filter=blob:none",
                    "https://github.com/ggml-org/llama.cpp", ROOT], check=True)
try:
    subprocess.run(["git", "-C", ROOT, "fetch", "origin", COMMIT, "--depth", "1"], check=True)
    subprocess.run(["git", "-C", ROOT, "checkout", "-f", COMMIT], check=True)
    head = subprocess.run(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"],
                          capture_output=True, text=True).stdout.strip()
    print("pinned llama.cpp:", head)
except Exception as e:
    print("WARN: не удалось закрепить коммит, используем master:", e)


In [ ]:
import os, subprocess

# --- 2b. Конфигурация и сборка (llama-server + llama-bench) -------
ROOT = os.path.abspath("llama.cpp")
BUILD = os.path.join(ROOT, "cuda")
os.makedirs(BUILD, exist_ok=True)

cfg = subprocess.run(
    ["cmake", "-S", ROOT, "-B", BUILD,
     "-DGGML_CUDA=ON", "-DGGML_CPU=ON", "-DCMAKE_BUILD_TYPE=Release",
     "-DCMAKE_CUDA_ARCHITECTURES=75"],
    capture_output=True, text=True)
print("configure:", "OK" if cfg.returncode == 0 else "FAIL")
print(cfg.stderr[-400:] if cfg.returncode else cfg.stdout[-300:])

bld = subprocess.run(
    ["cmake", "--build", BUILD, "--target", "llama-server", "llama-bench",
     "-j", str(os.cpu_count())],
    capture_output=True, text=True)
print("build:", "OK" if bld.returncode == 0 else "FAIL")
print(bld.stderr[-600:] if bld.returncode else bld.stdout[-300:])


### 3. Модель

Тот же `Qwen2.5-0.5B-Instruct` GGUF, квантизация **Q4_K_M** (~0.4 ГБ) — файл с
официального репозитория Qwen, совпадает с локальным `model_repository/generation/1/model.gguf`.


In [ ]:
import os

from huggingface_hub import hf_hub_download

MODEL_REPO = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
MODEL_FILE = "qwen2.5-0.5b-instruct-q4_k_m.gguf"
MODEL_PATH = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE)
print(MODEL_PATH)
print(f"{os.path.getsize(MODEL_PATH) / 1e6:.1f} MB")


### 4. `llama-bench` — ток/с (pp512 / tg128)

Тот же вызов, что в локальных замерах: `-m <gguf> -t 4 -p 512 -n 128 -r 3 -o md`.


In [ ]:
import re, subprocess

BENCH_BIN = os.path.join(ROOT, "cuda", "bin", "llama-bench")
r = subprocess.run([BENCH_BIN, "-m", MODEL_PATH, "-t", "4",
                    "-p", "512", "-n", "128", "-r", "3", "-o", "md"],
                   capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print(r.stderr[-1500:])

def _val(out, key):
    m = re.search(re.escape(key) + r"\s*\|\s*([0-9.]+)", out)
    return float(m.group(1)) if m else float("nan")

T4_PP512 = _val(r.stdout, "pp512")
T4_TG128 = _val(r.stdout, "tg128")
print(f"parsed -> pp512={T4_PP512:.2f} tok/s, tg128={T4_TG128:.2f} tok/s")


### 5. `llama-server` — те же флаги, что в `perf/start_llama_server.sh`

`-ngl 99 -c 512 -np 4 -t 4 --port 8080` (на T4 это CUDA-бэкенд вместо Vulkan).


In [ ]:
import json, os, subprocess, time, urllib.request

SERVER = os.path.join(ROOT, "cuda", "bin", "llama-server")
PORT = 8080
LOGDIR = "/content"

# перезапуск на случай повторного запуска этой ячейки
subprocess.run(["pkill", "-f", "llama-server"], capture_output=True, check=False)
time.sleep(1)

server_log = open(os.path.join(LOGDIR, "llama_server.log"), "w")
proc = subprocess.Popen(
    [SERVER, "-m", MODEL_PATH, "-ngl", "99", "-c", "512", "-np", "4", "-t", "4",
     "--host", "0.0.0.0", "--port", str(PORT)],
    stdout=server_log, stderr=subprocess.STDOUT)
print("llama-server pid:", proc.pid)

for _ in range(120):
    try:
        with urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=2) as resp:
            if json.loads(resp.read()).get("status") == "ok":
                print("server ready"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("llama-server не поднялся: " + open("llama_server.log").read()[-1500:])


### 6. Smoke: один запрос через `/completion`

Формат тела — как в `model_repository/generation/1/model.py` (`n_predict=128`,
`temperature=0.1`, `cache_prompt=True`). Заодно видим чистую задержку llama.cpp
(без Triton), сопоставимую с `compute-infer` из метрик Triton.


In [ ]:
import time

TEXT = "What is machine learning and how does it work?"

def completion(prompt, n_predict=128, temperature=0.1):
    body = json.dumps({"prompt": prompt, "n_predict": n_predict,
                       "temperature": temperature, "cache_prompt": True}).encode()
    req = urllib.request.Request(f"http://localhost:{PORT}/completion",
                                 data=body, headers={"Content-Type": "application/json"})
    t0 = time.perf_counter()
    with urllib.request.urlopen(req, timeout=120) as resp:
        data = json.loads(resp.read().decode())
    ms = (time.perf_counter() - t0) * 1000
    return data["content"], ms, data.get("timings", {})

content, ms, timings = completion(TEXT)
print(f"single /completion: {ms:.0f} ms")
if timings:
    print(f"  tokens: {timings.get('predicted_n')}, "
          f"tg {timings.get('predicted_per_second', 0):.1f} tok/s")
print("answer:", content[:220].replace("\n", " "))


### 7. Load-test — зеркало `perf/load_test.py`

Те же параметры, что в `REPORT.md` раздел 4.2: `--model generation`
концептуально (`concurrency=2, duration=30, warmup=2`), но вместо gRPC в Triton
шлём HTTP `/completion` напрямую в llama-server (это ровно тот вызов, который
делал Triton-бэкенд `generation`).


In [ ]:
import statistics, threading, time

class LoadTest:
    def __init__(self, n_predict=128, temperature=0.1):
        self.n_predict, self.temperature = n_predict, temperature
        self._lat, self._lock, self._stop = [], threading.Lock(), threading.Event()

    def _send(self):
        body = json.dumps({"prompt": TEXT, "n_predict": self.n_predict,
                           "temperature": self.temperature, "cache_prompt": True}).encode()
        req = urllib.request.Request(f"http://localhost:{PORT}/completion",
                                     data=body, headers={"Content-Type": "application/json"})
        t0 = time.perf_counter()
        with urllib.request.urlopen(req, timeout=300):
            pass
        with self._lock:
            self._lat.append((time.perf_counter() - t0) * 1000)

    def _worker(self):
        while not self._stop.is_set():
            try:
                self._send()
            except Exception as e:
                print("worker error:", e); time.sleep(0.2)

    def run(self, concurrency=2, duration=30, warmup=2):
        time.sleep(warmup)
        workers = [threading.Thread(target=self._worker) for _ in range(concurrency)]
        started = time.time()
        for w in workers: w.start()
        deadline = started + duration
        while time.time() < deadline:
            time.sleep(0.05)
        self._stop.set()
        for w in workers: w.join()
        elapsed = time.time() - started
        with self._lock:
            lat = sorted(self._lat)
        return {
            "backend": "T4 CUDA (llama-server /completion)",
            "concurrency": concurrency,
            "elapsed_s": round(elapsed, 2),
            "requests": len(lat),
            "rps": round(len(lat) / elapsed, 2) if elapsed else 0.0,
            "mean_ms": round(statistics.mean(lat), 2) if lat else 0.0,
            "p50_ms": round(lat[len(lat) // 2], 2) if lat else 0.0,
            "p95_ms": round(lat[int(len(lat) * 0.95)], 2) if lat else 0.0,
            "p99_ms": round(lat[int(len(lat) * 0.99)], 2) if lat else 0.0,
        }

res_t4 = LoadTest().run(concurrency=2, duration=30, warmup=2)
print(json.dumps(res_t4, indent=2))


### 8. Сравнение с локальными замерами

Локальные цифры (последний прогон, тот же GGUF, тот же коммит llama.cpp) —
при необходимости замените `LOCAL_*` своими.


In [ ]:
# Данные из локального прогона (REPORT.md, разделы 4.2–4.3)
from pathlib import Path

LOCAL_BENCH = {   # tok/s, llama-bench -t 4 -p 512 -n 128 -r 3
    "CPU (AMD, -t4)":    {"pp512": 543.96,  "tg128": 99.13},
    "Vulkan (780M)":     {"pp512": 4604.77, "tg128": 123.28},
    # "T4 (CUDA)": {"pp512": T4_PP512, "tg128": T4_TG128},
}
LOCAL_GEN = {     # generation 128 токенов, concurrency 2, duration 30
    "CPU (через Triton)":    {"rps": 0.72,  "p50_ms": 2703.09, "p95_ms": 3685.14},
    "Vulkan (через Triton)": {"rps": 1.13, "p50_ms": 1948.93, "p95_ms": 2789.89},
}

print("\n### llama-bench, ток/с\n")
print("| Backend | pp512 | tg128 |")
print("|---------|-------|-------|")
for name, v in {**LOCAL_BENCH,
                "T4 CUDA (-t4)": {"pp512": T4_PP512, "tg128": T4_TG128}}.items():
    print(f"| {name} | {v['pp512']:.1f} | {v['tg128']:.1f} |")

print("\n### generation, 128 токенов (load-test concurrency=2, 30 s)\n")
print("| Вариант | rps | p50 ms | p95 ms |")
print("|---------|-----|--------|--------|")
for name, v in {**LOCAL_GEN,
                "T4 CUDA (/completion напрямую)": res_t4}.items():
    print(f"| {name} | {v['rps']} | {v['p50_ms']:.2f} | {v['p95_ms']:.2f} |")

# Сохраняем результат как artefacts
Path("/content/results_t4.json").write_text(json.dumps(res_t4, indent=2))
print("\nsaved /content/results_t4.json")


### 9. Как читать результат (связь с разделом 4)

- **pp512** (prefill) — если T4-значение заметно выше 4605 ток/с Vulkan-замера,
  это ожидаемо: T4 (FP16, ~65 TFLOPS) сильнее iGPU 780M (~2.8 TFLOPS).
- **tg128** (autoregressive) — у мелкой 0.5B модели генерация упирается не в
  «сырые» TFLOPS, а в память/декодирование, поэтому T4 даст рост умереннее.
- **p50 через `/completion`** сопоставим с `compute-infer` метрик Triton
  (локально: 2048 мс CPU / 1416 мс Vulkan); в Triton ещё добавляется очередь и
  HTTP-хоп, поэтому полная генерация в репорте была 2703/1949 мс.
- Строки `LOCAL_*` в ячейке 8 можно править, чтобы вставить свои прогоны.

> Optional: чтобы повторить и шаги `classification`/`embedding`/`ensemble`
> (ONNX Runtime + CUDA) на той же модели — раскомментируйте следующий раздел.


### 10. Optional: ONNX-стадии (classification / embedding) на T4

Повторяет `export_models.py` + стадии BLS из `ensemble/1/model.py`
(tokenizer → embedding + classification → routing). Нужен `onnxruntime-gpu`
и `optimum` (ставится ~1–2 минуты). Пропустите эту ячейку — необязательно.


In [ ]:
# %% optional
import os, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# ставим GPU-вариант ORT (CPU-версия часто уже стоит в Colab)
subprocess.run(["pip", "install", "-q", "--upgrade", "onnxruntime-gpu"], check=False)
import onnxruntime as ort
providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
print("ORT available providers:", ort.get_available_providers())
print("fallback to CPU EP допустим — стадии лёгкие")

from transformers import AutoTokenizer
from optimum.onnxruntime import ORTModelForSequenceClassification, ORTModelForFeatureExtraction

# ровно как export_models.py
cls_dir, emb_dir = "onnx/classification", "onnx/embedding"
os.makedirs(cls_dir, exist_ok=True); os.makedirs(emb_dir, exist_ok=True)
tok_cls = AutoTokenizer.from_pretrained("huawei-noah/TinyBERT_General_4L_312D")
tok_cls.save_pretrained(cls_dir)
cls_model = ORTModelForSequenceClassification.from_pretrained("huawei-noah/TinyBERT_General_4L_312D", export=True)
cls_model.save_pretrained(cls_dir)
tok_emb = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
tok_emb.save_pretrained(emb_dir)
emb_model = ORTModelForFeatureExtraction.from_pretrained("sentence-transformers/all-MiniLM-L6-v2", export=True)
emb_model.save_pretrained(emb_dir)
print("ONNX exported to", cls_dir, emb_dir)


In [ ]:
# %% optional — BLS-стадии на ONNX Runtime (GPU EP)
import numpy as np, time

sess_cls = ort.InferenceSession(os.path.join(cls_dir, "model.onnx"), providers=providers)
sess_emb = ort.InferenceSession(os.path.join(emb_dir, "model.onnx"), providers=providers)

def encode(tok, text, max_length=128):
    e = tok(text, padding="max_length", truncation=True, max_length=max_length,
            return_tensors="np")
    return {k: v.astype(np.int64) for k, v in e.items()}

def classify(text):
    inp = encode(tok_cls, text)
    t0 = time.perf_counter()
    logits = sess_cls.run(["logits"], dict(inp))[0][0]
    probs = np.exp(logits - logits.max()); probs /= probs.sum()
    return int(np.argmax(probs)), float(probs.max()), (time.perf_counter() - t0) * 1000

def embedding(text):
    inp = encode(tok_emb, text)
    t0 = time.perf_counter()
    hidden = sess_emb.run(["last_hidden_state"], dict(inp))[0][0]
    mask = inp["attention_mask"][0, :, None].astype(np.float32)
    pooled = (hidden * mask).sum(0) / max(mask.sum(), 1e-9)
    pooled = pooled / np.linalg.norm(pooled)
    return pooled, (time.perf_counter() - t0) * 1000

label, conf, t_cls = classify(TEXT)
vec, t_emb = embedding(TEXT)
print(f"classification: label={label}, conf={conf:.3f}, {t_cls:.1f} ms")
print(f"embedding:      dim={vec.shape[0]}, {t_emb:.1f} ms")
if label == 1:
    content, ms, _ = completion(TEXT)
    print(f"→ вопрос → generation: {ms:.0f} ms")
    print("  answer:", content[:160].replace("\n", " "))
else:
    print("→ утверждение → classification-only path")
